# Hybrid Model Evaluation & Hypothesis Testing

In `graph_tree_hybrid.ipynb`, we obtained graph embeddings ($D = 16$) from a Deep Neural Decision Forest, where the hidden layers are based on Graph GPS and the final layer is a decision tree with maximum depth 5. We showed that concatenating these embeddings with the original node features to form hybrid features leads to improved performance: a Random Forest trained on the hybrid features outperforms one trained on the node features alone.

In this notebook, we aim to demonstrate that this improvement is not a one-time occurrence, but rather that the learned graph embeddings are consistently effective in enhancing model performance.

**Note:** The CSV file recording the performance of both hybrid and non-hybrid Random Forests is saved as `rf_performences.csv`. One can run this notebook starting from the “Evaluating Performance” section.

In [2]:
import torch
from torch_geometric.data import Data

elliptic = torch.load("../Dataset/processed/transaction_graph_v1.pt", weights_only=False, map_location="cpu")
txs_data = Data(**elliptic[0])
txs_data

Data(x=[203769, 166], edge_index=[2, 234355], y=[203769], train_mask=[203769], val_mask=[203769], test_mask=[203769])

In [ ]:
# Implement strict time-forward train/val/test split
time_steps = txs_data.x[:,0]
train_mask = (time_steps >= 1) & (time_steps <= 32)
val_mask = (time_steps >= 33) & (time_steps <= 37)
test_mask = (time_steps >= 38) & (time_steps <= 42)

# Select the labeled data and make the train/val/test sets
labeled = txs_data.y != 2
train_idx = train_mask & labeled
val_idx = val_mask & labeled
test_idx = test_mask & labeled

# Retrieve embedding matrix and construct hybrid features
h = torch.load("./GNNs/embeddings/dndf_embeddings.pt")

# Prepare training/validation/testing data
X_node = txs_data.x[:,1:]
X_train_node = txs_data.x[train_idx, 1:]
X_val_node = txs_data.x[val_idx, 1:]
X_test_node = txs_data.x[test_idx, 1:]

X_hybrid = torch.cat([X_node, h], dim=1)
X_train_hybrid = X_hybrid[train_idx]
X_val_hybrid = X_hybrid[val_idx]
X_test_hybrid = X_hybrid[test_idx]

y = txs_data.y
y_train = txs_data.y[train_idx]
y_val = txs_data.y[val_idx]
y_test = txs_data.y[test_idx]


print("The size of the training set is:", train_idx.sum().item())
print("The size of the validation set is:",val_idx.sum().item())
print("The size of the testing set is:", test_idx.sum().item())

The size of the training set is: 28938
The size of the validation set is: 4503
The size of the testing set is: 6436


## Trainining Hybrid and Non-hybrid Random Forests

We construct a set of Random Forest configurations in which `max_depth = 5` is fixed while all other hyperparameters vary. For each configuration, we first train the model on node-only features and then on hybrid features. We record the performance metric (PR-AUC) on the training, validation, and testing sets.

In [ ]:
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve, auc

# Generate a list of model parameters (fix max_depth = 5)
param_grid = {
    'class_weight': [None, 'balanced'],
    'max_depth': [5],
    'max_features': ['sqrt', 0.3, 0.5],
    'min_samples_leaf': [1, 2, 5],
    'min_samples_split': [2, 5, 10],
    'n_estimators': [50]
}
params = list(ParameterGrid(param_grid))
rows = []
# For each set of hyperparameter, train the random forest on both node-only features and hybrid features
for param in tqdm(params):

  if param['min_samples_split'] < 2 * param['min_samples_leaf']:
    continue

  rf = RandomForestClassifier(**param, random_state=42)

  # Train on node-only features and get performance metrics
  rf.fit(X_train_node, y_train)
  y_scores_node = rf.predict_proba(X_node)[:,-1]
  train_precision_node, train_recall_node, _ = precision_recall_curve(y_train, y_scores_node[train_idx])
  val_precision_node, val_recall_node, _ = precision_recall_curve(y_val, y_scores_node[val_idx])
  test_precision_node, test_recall_node, _ = precision_recall_curve(y_test, y_scores_node[test_idx])

  # Train on hybrid features and get performance metrics
  rf.fit(X_train_hybrid, y_train)
  y_scores_hybrid = rf.predict_proba(X_hybrid)[:,-1]
  train_precision_hybrid, train_recall_hybrid, _ = precision_recall_curve(y_train, y_scores_hybrid[train_idx])
  val_precision_hybrid, val_recall_hybrid, _ = precision_recall_curve(y_val, y_scores_hybrid[val_idx])
  test_precision_hybrid, test_recall_hybrid, _ = precision_recall_curve(y_test, y_scores_hybrid[test_idx])

  # Store the performance metrics
  row = {
        ('pr-auc', 'train (node only)'): auc(train_recall_node, train_precision_node),
        ('pr-auc', 'train (hybrid)'): auc(train_recall_hybrid, train_precision_hybrid),
        ('pr-auc', 'val (node only)'): auc(val_recall_node, val_precision_node),
        ('pr-auc', 'val (hybrid)'): auc(val_recall_hybrid, val_precision_hybrid),
        ('pr-auc', 'test (node only)'): auc(test_recall_node, test_precision_node),
        ('pr-auc', 'test (hybrid)'): auc(test_recall_hybrid, test_precision_hybrid),
    }
  rows.append(row)

100%|██████████| 54/54 [25:59<00:00, 28.89s/it]


In [ ]:
# Build dataframe for model performance
perf_df = pd.DataFrame(rows)
perf_df.columns = pd.MultiIndex.from_tuples(perf_df.columns)

# Build dataframe for model parameters
filtered_params = [param for param in params if param['min_samples_split'] >= 2 * param['min_samples_leaf']]
params_df = pd.DataFrame(filtered_params)

# Combine the two dataframes
df = pd.concat([params_df, perf_df], axis=1)
df.columns = [
    'class_weight', 'max_depth', 'max_features', 'leaf', 'split', 'n_estimators',
    'train_pr_auc (node)', 'train_pr_auc (hybrid)',
    'val_pr_auc (node)', 'val_pr_auc (hybrid)',
    'test_pr_auc (node)', 'test_pr_auc (hybrid)'
]
df = df.round(4)
df.to_csv("./GNNs/rf_performances.csv", index=False)

## Evaluating performance


We read the csv file which stores the performance metrics of hybrid and non-hybrid Random Forest models.

In [4]:
import pandas as pd 
df = pd.read_csv('../gnn/rf_performances.csv')
df.head(7)

,class_weight,max_depth,max_features,leaf,split,n_estimators,train_pr_auc (node),train_pr_auc (hybrid),val_pr_auc (node),val_pr_auc (hybrid),test_pr_auc (node),test_pr_auc (hybrid)
0,NaN,5,sqrt,1,2,50,0.9736,0.9740,0.9331,0.9329,0.8845,0.8880
1,NaN,5,sqrt,1,5,50,0.9735,0.9735,0.9326,0.9332,0.8842,0.8886
2,NaN,5,sqrt,1,10,50,0.9724,0.9732,0.9310,0.9334,0.8817,0.8880
3,NaN,5,sqrt,2,5,50,0.9734,0.9732,0.9336,0.9325,0.8844,0.8886
4,NaN,5,sqrt,2,10,50,0.9723,0.9730,0.9320,0.9329,0.8818,0.8880
5,NaN,5,sqrt,5,10,50,0.9715,0.9729,0.9288,0.9326,0.8812,0.8898
6,NaN,5,0.3,1,2,50,0.9726,0.9743,0.9164,0.9289,0.8780,0.8880


We construct a new dataframe which computes the PR-AUC gains for each model.

In [5]:
gain= pd.DataFrame({
    'train_gain': df['train_pr_auc (hybrid)'] - df['train_pr_auc (node)'],
    'val_gain': df['val_pr_auc (hybrid)'] - df['val_pr_auc (node)'],
    'test_gain': df['test_pr_auc (hybrid)'] - df['test_pr_auc (node)'],
})
gain['train_gain_pct'] = gain['train_gain'] / df['train_pr_auc (node)'] * 100
gain['val_gain_pct']   = gain['val_gain']   / df['val_pr_auc (node)'] * 100
gain['test_gain_pct']  = gain['test_gain']  / df['test_pr_auc (node)'] * 100
gain = gain.round(4)

gain.head(7)

,train_gain,val_gain,test_gain,train_gain_pct,val_gain_pct,test_gain_pct
0,0.0004,-0.0002,0.0035,0.0411,-0.0214,0.3957
1,0.0000,0.0006,0.0044,0.0000,0.0643,0.4976
2,0.0008,0.0024,0.0063,0.0823,0.2578,0.7145
3,-0.0002,-0.0011,0.0042,-0.0205,-0.1178,0.4749
4,0.0007,0.0009,0.0062,0.0720,0.0966,0.7031
5,0.0014,0.0038,0.0086,0.1441,0.4091,0.9759
6,0.0017,0.0125,0.0100,0.1748,1.3640,1.1390


We display some gain statistics.

In [6]:
print(f"We have tested {len(gain)} models in total.")
print(f"On the training set, the hybrid model outperforms the node-only model {int((gain['train_gain'] > 0).sum())} times. The average percentage gain is {gain['train_gain_pct'].mean():.4f}%.")
print(f"On the validation set, the hybrid model outperforms the node-only model {int((gain['val_gain'] > 0).sum())} times. The average percentage gain is {gain['val_gain_pct'].mean():.4f}%.")
print(f"On the testing set, the hybrid model outperforms the node-only model {int((gain['test_gain'] > 0).sum())} times. The average percentage gain is {gain['test_gain_pct'].mean():.4f}%.")

We have tested 36 models in total.
On the training set, the hybrid model outperforms the node-only model 16 times. The average percentage gain is 0.0832%.
On the validation set, the hybrid model outperforms the node-only model 16 times. The average percentage gain is 0.1935%.
On the testing set, the hybrid model outperforms the node-only model 36 times. The average percentage gain is 0.8969%.


As we can see, while the hybrid models show only minor improvements on the training and validation sets, they show consistent improvements on the test set across all hyperparameters. This suggests that the embedding matrix captures important topological information of the graph. To make this more rigorous (and to account for the magnitude of the improvements), we perform hypothesis testing as follows.

## Hypothesis Testing

We want to test the null hypothesis

\begin{equation*}
H_0: \text{PR-AUC}_{\text{hybrid}} \leq \text{PR-AUC}_{\text{node-only}}
\end{equation*}

and the alternative hypothesis

\begin{equation*}
H_1: \text{PR-AUC}_{\text{hybrid}} \geq \text{PR-AUC}_{\text{node-only}}
\end{equation*}

on the training, validation, and testing sets. To do this, we use a paired $t$-test. For each model $(i = 1, \dots, 36)$, we define the paired difference

\begin{equation*}
\Delta_i = \text{PR-AUC}_{\text{hybrid}}^{i} - \text{PR-AUC}_{\text{node-only}}^{i}.
\end{equation*}

The null hypothesis can then be written as $H_0: \mu \le 0$, where $\mu = \mathbb{E}[\Delta]$ is the mean difference.

Let $\bar{\Delta}$ and $s$ denote the sample mean and sample standard deviation of $\{\Delta_{i}\}_{i=1}^{36}$. Our test statistic is
\begin{equation*}
t = \frac{\bar{\Delta}}{s / \sqrt{36}} = \frac{6 \bar{\Delta}}{s},
\end{equation*}
which, under mild assumptions, follows a Student’s $t$-distribution with $35$ degrees of freedom. We compute the corresponding $p$-value and reject $H_0$ if it is below a chosen significance level $\alpha$ (e.g., $\alpha = 0.05$).


In [7]:
# Compute test statistics and p-values
from scipy.stats import ttest_rel

t_train, p_train = ttest_rel(df['train_pr_auc (hybrid)'], df['train_pr_auc (node)'],alternative='greater')
t_val, p_val = ttest_rel(df['val_pr_auc (hybrid)'], df['val_pr_auc (node)'],alternative='greater')
t_test, p_test = ttest_rel(df['test_pr_auc (hybrid)'], df['test_pr_auc (node)'],alternative='greater')

print(f"Train t-statistic: {t_train:.4f}. Val t-statistic: {t_val:.4f}. Test t-statistic: {t_test:.4f}.")
print(f"Train p-value: {p_train:.4f}. Val p-value: {p_val: .4f}. Test p-value: {p_test: .4f}.")

Train t-statistic: 2.7170. Val t-statistic: 1.6750. Test t-statistic: 8.9009.
Train p-value: 0.0051. Val p-value:  0.0514. Test p-value:  0.0000.


In [8]:
# Perform hypothesis testing
alpha_train, alpha_val, alpha_test = 0.01, 0.06, 0.01

if p_train < alpha_train:
    print(f"On the training set, we reject H0 at significance level α = {alpha_train}")
else:
    print(f"On the training set, we fail to reject H0 at significance level α = {alpha_train}")

if p_val < alpha_val:
    print(f"On the validation set, we reject H0 at significance level α = {alpha_val}")
else:
    print(f"On the validation set, we fail to reject H0 at significance level α = {alpha_val}")

if p_test < alpha_test:
    print(f"On the testing set, we reject H0 at significance level α = {alpha_test}")
else:
    print(f"On the testing set, we fail to reject H0 at significance level α = {alpha_test}")

On the training set, we reject H0 at significance level α = 0.01
On the validation set, we reject H0 at significance level α = 0.06
On the testing set, we reject H0 at significance level α = 0.01
